# Segmentting customers with Python

In [ ]:
# Loading the data
import pandas as pd
import janitor
df = pd.read_excel('../../data/raw/Online Retail.xlsx', sheet_name='Online Retail').clean_names(case_type='snake')

In [ ]:
df.shape

In [ ]:
df.head()

# Data Clean-Up

#### - Negative Quantity

In [ ]:
df.loc[df['quantity'] <= 0].shape

In [ ]:
df.shape

In [ ]:
df = df.loc[df['quantity'] > 0]

In [ ]:
df.shape

#### - Missing CustomerID

In [ ]:
pd.isnull(df['customer_id']).sum()

In [ ]:
df.shape

In [ ]:
df = df[pd.notnull(df['customer_id'])]

In [ ]:
df.shape

In [ ]:
df.head()

#### - Excluding Incomplete Month

In [ ]:
print('Date Range: %s ~ %s' % (df['invoice_date'].min(), df['invoice_date'].max()))

In [ ]:
df.loc[df['invoice_date'] >= '2011-12-01'].shape

In [ ]:
df.shape

In [ ]:
df = df.loc[df['invoice_date'] < '2011-12-01']

In [ ]:
df.shape

#### - Total Sales

In [ ]:
df['sales'] = df['quantity'] * df['unit_price']

In [ ]:
df.head()

#### - Per Customer Data

In [ ]:
customer_df = df.groupby('customer_id').agg({
    'sales': sum,
    'invoice_no': lambda x: x.nunique()
})

customer_df.columns = ['total_sales', 'order_count']
customer_df['avg_order_value'] = customer_df['total_sales']/customer_df['order_count']

In [ ]:
customer_df.head(15)

In [ ]:
customer_df.describe()

In [ ]:
rank_df = customer_df.rank(method='first')

In [ ]:
rank_df.head(15)

In [ ]:
rank_df.describe()

In [ ]:
normalized_df = (rank_df - rank_df.mean()) / rank_df.std()

In [ ]:
normalized_df.head(15)

In [ ]:
normalized_df.describe()

# Customer Segmentation via K-Means Clustering

In [ ]:
from sklearn.cluster import KMeans

#### - K-Means Clustering

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42).fit(normalized_df[['total_sales', 'order_count', 'avg_order_value']])

In [ ]:
kmeans

In [ ]:
kmeans.labels_

In [ ]:
kmeans.cluster_centers_

In [ ]:
four_cluster_df = normalized_df[['total_sales', 'order_count', 'avg_order_value']].copy(deep=True)
four_cluster_df['cluster'] = kmeans.labels_

In [ ]:
four_cluster_df.head()

In [ ]:
four_cluster_df.groupby('cluster').count()['total_sales']

In [ ]:
from plotly.subplots import make_subplots

import plotly.graph_objects as go

colors = {0: 'blue', 1: 'red', 2: 'orange', 3: 'green'}
cluster_names = {0: 'Cluster 0', 1: 'Cluster 1', 2: 'Cluster 2', 3: 'Cluster 3'}

fig = make_subplots(rows=1, cols=3, subplot_titles=(
    'TotalSales vs. OrderCount Clusters',
    'AvgOrderValue vs. OrderCount Clusters',
    'AvgOrderValue vs. TotalSales Clusters'
))

for cluster in range(4):
    cluster_data = four_cluster_df[four_cluster_df['cluster'] == cluster]
    
    fig.add_trace(go.Scatter(
        x=cluster_data['order_count'],
        y=cluster_data['total_sales'],
        mode='markers',
        marker=dict(color=colors[cluster]),
        name=cluster_names[cluster],
        legendgroup=cluster_names[cluster],
        showlegend=True
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=cluster_data['order_count'],
        y=cluster_data['avg_order_value'],
        mode='markers',
        marker=dict(color=colors[cluster]),
        name=cluster_names[cluster],
        legendgroup=cluster_names[cluster],
        showlegend=False
    ), row=1, col=2)

    fig.add_trace(go.Scatter(
        x=cluster_data['total_sales'],
        y=cluster_data['avg_order_value'],
        mode='markers',
        marker=dict(color=colors[cluster]),
        name=cluster_names[cluster],
        legendgroup=cluster_names[cluster],
        showlegend=False
    ), row=1, col=3)

fig.update_xaxes(title_text='Order Count', row=1, col=1)
fig.update_yaxes(title_text='Total Sales', row=1, col=1)
fig.update_xaxes(title_text='Order Count', row=1, col=2)
fig.update_yaxes(title_text='Avg Order Value', row=1, col=2)
fig.update_xaxes(title_text='Total Sales', row=1, col=3)
fig.update_yaxes(title_text='Avg Order Value', row=1, col=3)

fig.update_layout(
    template='ggplot2',
    title_text='Cluster Analysis',
    height=500,
    width=1400
)

fig.show()


#### - Selecting the best number of clusters

In [ ]:
from sklearn.metrics import silhouette_score

In [ ]:
for n_cluster in [4,5,6,7,8]:
    kmeans = KMeans(n_clusters=n_cluster).fit(
        normalized_df[['total_sales', 'order_count', 'avg_order_value']]
    )
    silhouette_avg = silhouette_score(
        normalized_df[['total_sales', 'order_count', 'avg_order_value']], 
        kmeans.labels_
    )
    
    print('Silhouette Score for %i Clusters: %0.4f' % (n_cluster, silhouette_avg))

#### - Interpreting Customer Segments

In [ ]:
kmeans = KMeans(n_clusters=4).fit(
    normalized_df[['total_sales', 'order_count', 'avg_order_value']]
)

In [ ]:
four_cluster_df = normalized_df[['total_sales', 'order_count', 'avg_order_value']].copy(deep=True)
four_cluster_df['cluster'] = kmeans.labels_

In [ ]:
four_cluster_df.head(15)

In [ ]:
kmeans.cluster_centers_

In [ ]:
high_value_cluster = four_cluster_df.loc[four_cluster_df['cluster'] == 2]
high_value_cluster.head()

In [ ]:
customer_df.loc[high_value_cluster.index].describe()

In [ ]:
pd.DataFrame(
    df.loc[
        df['customer_id'].isin(high_value_cluster.index)
    ].groupby('description').count()[
        'stock_code'
    ].sort_values(ascending=False).head()
)

In [ ]:
pd.DataFrame(
    df.loc[
        df['customer_id'].isin(
            four_cluster_df.loc[four_cluster_df['cluster'] == 1].index
        )
    ].groupby('description').count()[
        'stock_code'
    ].sort_values(ascending=False).head()
)